# 09 — Search Results: Coverage and Language Distribution

Analysis of the GitHub search results produced by running the multilingual search pipeline against the reviewed search terms from `reviewed_grouped_translated_terms.csv`.

Three result types are available: **repositories**, **issues**, and **users**. Each was searched by term string; results carry `search_term`, `natural_language` (the comma-separated language codes that share that term), and standard GitHub API fields.

This notebook asks:

- How many results does each search type return, and how many unique items survive deduplication?
- What fraction of the 4,285 kept terms returned any results at all?
- How many pipeline languages have at least one result of any type?
- **What does the multilingual approach add over English-only or a handful of major languages?**
- Which language families drive the most results, and which are only reachable through non-English terms?

**Sections**
1. Data loading and deduplication
2. Search term hit rate
3. The value of multilingual search
4. Search-term overlap and retrieval clusters
5. Vocabulary-source ablation
6. Language and family coverage
7. Result distribution by language family

In [1]:
import os
import sys
import glob
import unicodedata

import pandas as pd
import altair as alt
import networkx as nx
from pathlib import Path

alt.data_transformers.enable("vegafusion")

sys.path.insert(0, str(Path("..").resolve()))
from scripts.utils import get_data_directory_path, get_language_family, save_chart

DATA_DIR     = get_data_directory_path()
TERM         = "Digital Humanities"
TERM_SLUG    = TERM.lower().replace(" ", "_")
FIGURE_DIR   = Path("..") / "figures" / "dhq_article"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Search results live outside the pipeline datasets directory
SEARCH_DIR   = Path(DATA_DIR).parent.parent / "new_datasets"
REPO_GLOB    = str(SEARCH_DIR / "searched_repo_data"  / "digital_humanities" / "*.csv")
ISSUE_GLOB   = str(SEARCH_DIR / "searched_issue_data" / "digital_humanities" / "*.csv")
USER_GLOB    = str(SEARCH_DIR / "searched_user_data"  / "digital_humanities" / "*.csv")

SEARCH_TERMS_PATH = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG,
    "search_terms", "reviewed_grouped_translated_terms.csv"
)
SEARCH_RELEVANCE_PATH = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG,
    "search_terms", "search_result_relevance_reviewed.csv"
)
SEARCH_RELEVANCE_QUEUE_PATH = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG,
    "search_terms", "search_result_relevance_review.csv"
)
SEARCH_OVERLAP_PATH = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG,
    "search_terms", "search_result_term_overlap_edges.csv"
)
SEARCH_OVERLAP_SUMMARY_PATH = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG,
    "search_terms", "search_result_overlap_summary.csv"
)
DISAGR_PATH  = os.path.join(
    DATA_DIR, "translated_terms", TERM_SLUG, "evaluation", "disagreement_analysis.csv"
)

print(f"Search dir: {SEARCH_DIR}")
print(f"Search terms: {SEARCH_TERMS_PATH}")

Retrieving translation pipeline data directory path...

Search dir: /Users/zleblanc/CodingDH/new_datasets
Search terms: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/search_terms/reviewed_grouped_translated_terms.csv


In [2]:
# ── Manual exclusions (search notebook: both types apply) ───────────────────
from scripts.utils import load_manual_exclusions

excl_eval_dir = os.path.join(DATA_DIR, "translated_terms", TERM_SLUG, "evaluation")
analysis_terms, search_terms, corrections = load_manual_exclusions(excl_eval_dir)
print(f"Manual exclusions loaded:")
print(f"  analysis_exclusion : {len(analysis_terms)} (language, term) pairs  (terms excluded from analysis)")
print(f"  search_exclusion   : {len(search_terms)} (language, term) pairs  (also excluded from search)")
print(f"  term_correction    : {len(corrections)} corrections")
print()
print("Note: search results for analysis-excluded and search-excluded terms")
print("      are excluded without removing unrelated terms from the same language.")

Manual exclusions loaded:
  analysis_exclusion : 122 (language, term) pairs  (terms excluded from analysis)
  search_exclusion   : 383 (language, term) pairs  (also excluded from search)
  term_correction    : 94 corrections

Note: search results for analysis-excluded and search-excluded terms
      are excluded without removing unrelated terms from the same language.


## 8.1 — Data Loading and Deduplication

Each result type is spread across one CSV per search term. Files are concatenated and deduplicated by GitHub item `id`. The pre-dedup row count reflects how many terms found each item (a repo discovered by three different search terms appears three times before dedup).

In [3]:
def load_results(file_glob: str, id_col: str = "id") -> pd.DataFrame:
    files = glob.glob(file_glob)
    if not files:
        return pd.DataFrame()
    return pd.concat(
        [pd.read_csv(f, low_memory=False) for f in files],
        ignore_index=True,
    )

repos_raw  = load_results(REPO_GLOB)
issues_raw = load_results(ISSUE_GLOB)
users_raw  = load_results(USER_GLOB)

# Filter to current reviewed terms only — excludes stale result files
kept_terms = set(
    pd.read_csv(SEARCH_TERMS_PATH)
    .query("keep_term == True")["search_term"]
    .str.lower().str.strip()
)

# Apply the post-search relevance audit when its reviewed artifact exists.
# Terms marked retain or retain_track remain in the analysis. Seeded exclusions are
# also retained until a human confirms them; only human-reviewed exclusions drop.
if os.path.exists(SEARCH_RELEVANCE_PATH):
    relevance_review = pd.read_csv(SEARCH_RELEVANCE_PATH, keep_default_na=False)
    if "review_status" in relevance_review.columns:
        confirmed_review = relevance_review["review_status"].eq("human_reviewed")
    else:
        # Backward compatibility: older reviewed files predate explicit review provenance.
        confirmed_review = pd.Series(True, index=relevance_review.index)
    excluded_retrieval_terms = set(
        relevance_review.loc[
            relevance_review["review_decision"].eq("exclude") & confirmed_review,
            "search_term_key",
        ].str.lower().str.strip()
    )
    kept_terms -= excluded_retrieval_terms
    print(f"Post-search relevance review: excluded {len(excluded_retrieval_terms)} human-confirmed terms")
else:
    excluded_retrieval_terms = set()
    print("Post-search relevance review not found; counts include all retained search terms.")

repos_raw  = repos_raw[repos_raw["search_term"].str.lower().str.strip().isin(kept_terms)]
issues_raw = issues_raw[issues_raw["search_term"].str.lower().str.strip().isin(kept_terms)]
users_raw  = users_raw[users_raw["search_term"].str.lower().str.strip().isin(kept_terms)]

repos  = repos_raw.drop_duplicates("id").copy()
issues = issues_raw.drop_duplicates("id").copy()
users  = users_raw.drop_duplicates("id").copy()

summary = pd.DataFrame([
    {"type": "Repositories", "raw_rows": len(repos_raw),  "unique_items": len(repos),
     "terms_that_hit": repos_raw["search_term"].nunique(),
     "avg_term_hits": round(len(repos_raw) / len(repos), 1)},
    {"type": "Issues",       "raw_rows": len(issues_raw), "unique_items": len(issues),
     "terms_that_hit": issues_raw["search_term"].nunique(),
     "avg_term_hits": round(len(issues_raw) / len(issues), 1)},
    {"type": "Users",        "raw_rows": len(users_raw),  "unique_items": len(users),
     "terms_that_hit": users_raw["search_term"].nunique(),
     "avg_term_hits": round(len(users_raw) / len(users), 1)},
])
print(summary.to_string(index=False))

Post-search relevance review not found; counts include all retained search terms.
        type  raw_rows  unique_items  terms_that_hit  avg_term_hits
Repositories     39115          6402              94            6.1
      Issues     34399          6862             143            5.0
       Users      5560          1696              33            3.3


In [4]:
repos.search_term.value_counts().head(20)

search_term
Digital Humanities        1808
Humanities Digital        1236
digital humanities         821
humanities digital         755
digitäl humäns             662
buku digital               235
informatica umanistica     154
Humanidades Digital        115
artes digitales            113
Humanités numériques        87
数字人文                        63
Humanités Numériques        55
humanidades digital         44
Digihumanitaaria            38
Humanidades Digitais        32
digita hūmanitī             30
數位人文                        13
humanités numériques        11
Humanidades digitales       11
humanidades digitais         8
Name: count, dtype: int64

In [5]:
users.search_term.value_counts().head(20)

search_term
Digital Humanities        1029
digital humanities         533
Informatica umanistica      30
Humanidades Digital         26
Humanidades Digitais        20
artes digitales             16
Humanidades digitales       11
informatica umanistica       6
humanidades digital          4
Humanités Numeriques         3
Humanidades digitais         3
humanidades digitales        3
dijital sanatlar             2
buku digital                 1
digital humanidades          1
Digital humaniora            1
Sastra Digital               1
sastra digital               1
Digitaalhumanitaaria         1
humanities digital           1
Name: count, dtype: int64

In [6]:
issues.search_term.value_counts().head(20)

search_term
Humanities Digital      1374
nhân văn số              894
digitäl humäns           827
digital a humanities     774
数字人文                     539
Digital Humanities       438
humanities digital       302
电子文学                     252
ŋɔɔŋ ŋɔɔŋ ŋɔɔŋ ŋɔɔŋ      166
人工智能文学                   128
buku digital             123
Nhân văn số              120
數字人文                      85
humanidades diġital       76
Humanités Numeriques      69
數字文學                      67
数位人文                      64
数字文化学                     52
Humanidades Digitais      30
humano digitais           27
Name: count, dtype: int64

In [7]:
# Load search terms and build helper maps
st = pd.read_csv(SEARCH_TERMS_PATH)
st_kept = st[st["keep_term"] == True].copy()
st_kept["search_term_lower"] = st_kept["search_term"].str.lower().str.strip()
st_kept = st_kept[~st_kept["search_term_lower"].isin(excluded_retrieval_terms)].copy()

disagr = pd.read_csv(DISAGR_PATH, converters={"language_code": str})

# Explode natural_language (comma-separated) → individual language codes
# Filters out multi-word artifacts like 'see also: test languages ...'
lang_term_rows = [] 
for _, row in st_kept.iterrows():
    codes = [
        c.strip() for c in str(row["natural_language"]).split(",")
        if c.strip() and " " not in c.strip()
    ]
    for code in codes:
        lang_term_rows.append({"language_code": code,
                               "search_term_lower": row["search_term_lower"],
                               "comparison_status": row["comparison_status"]})
lang_term_map = pd.DataFrame(lang_term_rows)

# Attach language family
lang_meta = disagr[["language_code", "language_name", "language_family"]].drop_duplicates("language_code")
lang_term_map = lang_term_map.merge(lang_meta, on="language_code", how="left")

print(f"Kept search terms: {len(st_kept)}")
print(f"Unique language codes mapped: {lang_term_map['language_code'].nunique()}")

Kept search terms: 4183
Unique language codes mapped: 876


## 8.2 — Search Term Hit Rate

Of the 4,285 kept terms, the vast majority return no GitHub results — which is expected. GitHub indexes public content in the language it was written; only a small fraction of the world's DH activity is on GitHub, and even less uses a term that matches a specific translation. The hit rate tells us which terms are productive and whether there is a long tail.

In [8]:
# Count results per search term across all types
def term_hit_counts(df: pd.DataFrame, label: str) -> pd.DataFrame:
    counts = (
        df.groupby(df["search_term"].str.lower().str.strip())["id"]
        .nunique()
        .reset_index()
    )
    counts.columns = ["search_term_lower", f"n_{label}"]
    return counts

repo_hits  = term_hit_counts(repos_raw,  "repos")
issue_hits = term_hit_counts(issues_raw, "issues")
user_hits  = term_hit_counts(users_raw,  "users")

hit_df = (
    st_kept[["search_term_lower", "comparison_status"]]
    .merge(repo_hits,  on="search_term_lower", how="left")
    .merge(issue_hits, on="search_term_lower", how="left")
    .merge(user_hits,  on="search_term_lower", how="left")
    .fillna(0)
)
for col in ["n_repos", "n_issues", "n_users"]:
    hit_df[col] = hit_df[col].astype(int)
hit_df["any_hit"] = (hit_df[["n_repos", "n_issues", "n_users"]] > 0).any(axis=1)
hit_df["total"]   = hit_df["n_repos"] + hit_df["n_issues"] + hit_df["n_users"]

print(f"Terms with ≥1 result of any type : {hit_df['any_hit'].sum()} / {len(hit_df)} ({hit_df['any_hit'].mean()*100:.1f}%)")
print(f"Terms with ≥1 repo               : {(hit_df['n_repos']>0).sum()}")
print(f"Terms with ≥1 issue              : {(hit_df['n_issues']>0).sum()}")
print(f"Terms with ≥1 user               : {(hit_df['n_users']>0).sum()}")
print()
print("Top 15 terms by total results:")
print(hit_df.nlargest(15, "total")[["search_term_lower","n_repos","n_issues","n_users","total"]].to_string(index=False))

Terms with ≥1 result of any type : 137 / 4183 (3.3%)
Terms with ≥1 repo               : 64
Terms with ≥1 issue              : 117
Terms with ≥1 user               : 18

Top 15 terms by total results:
    search_term_lower  n_repos  n_issues  n_users  total
   digital humanities     4691      2934     1562   9187
   humanities digital     2078      2890       19   4987
      digita hūmanitī     2066         0        0   2066
     humaniti digital     2063         0        0   2063
       digitäl humäns      666       829        0   1495
          nhân văn số        0      1018        0   1018
 digital a humanities        1       793        0    794
                 数字人文       75       565        0    640
         buku digital      235       123        1    359
humanidades digitales      180        75       46    301
  humanidades digital      176        76       31    283
                 电子文学        0       258        0    258
 humanités numériques      153        92        3    248
 h

In [9]:
# Distribution of result counts per term (zero-safe symlog scale for the long tail)
hist_data = pd.DataFrame({
    "type": ["Repos"] * len(hit_df) + ["Issues"] * len(hit_df) + ["Users"] * len(hit_df),
    "count": list(hit_df["n_repos"]) + list(hit_df["n_issues"]) + list(hit_df["n_users"]),
})
hist_data = hist_data[hist_data["count"] > 0]

base = alt.Chart(hist_data).mark_bar(opacity=0.7).encode(
    x=alt.X("count:Q", bin=alt.Bin(maxbins=40), title="results per term"),
    y=alt.Y("count():Q", title="number of terms",
            scale=alt.Scale(type="symlog", constant=1)),
    color=alt.Color("type:N", scale=alt.Scale(scheme="tableau10")),
).properties(width=200, height=160)

hist = base.facet(facet="type:N", columns=3,
                  title="Results per search term (symlog y-axis)")
display(hist)

alt.FacetChart(...)

### Top Productive Terms by Script Direction and Comparison Status

Which specific search terms drive the most results, and does script direction (LTR vs RTL) predict productivity? RTL scripts — Arabic, Hebrew, Persian, Urdu — represent communities that English-only search misses entirely. The string-level `comparison_status` records whether services converged exactly, differed only in near form, or produced substantively different terms. I compare search productivity across these descriptive statuses without treating any status as a guarantee of term quality.

In [10]:
# Merge directionality into hit_df
hit_dir = hit_df.merge(
    st_kept[["search_term_lower", "directionality"]].drop_duplicates("search_term_lower"),
    on="search_term_lower", how="left",
)
hit_dir["dir_label"] = hit_dir["directionality"].apply(
    lambda d: "RTL" if str(d).strip().lower() == "rtl" else "LTR"
)

# ── Top 25 terms: stacked repos / issues / users ──────────────────────────
top25 = hit_dir.nlargest(25, "total").copy()
top_order = top25.sort_values("total", ascending=False)["search_term_lower"].tolist()
top25_melt = top25.melt(
    id_vars=["search_term_lower", "dir_label", "total"],
    value_vars=["n_repos", "n_issues", "n_users"],
    var_name="result_type", value_name="n",
)
top25_melt["result_type"] = top25_melt["result_type"].str.replace("n_", "").str.capitalize()
type_order = ["Repos", "Issues", "Users"]

top_bar = alt.Chart(top25_melt).mark_bar().encode(
    y=alt.Y("search_term_lower:N",
            sort=top_order,
            title=None),
    x=alt.X("n:Q", title="results"),
    color=alt.Color("result_type:N", sort=type_order,
                    scale=alt.Scale(domain=type_order, scheme="tableau10"),
                    title="Result type"),
    order=alt.Order("result_type:N", sort="ascending"),
    tooltip=[
        alt.Tooltip("search_term_lower:N", title="term"),
        alt.Tooltip("dir_label:N", title="direction"),
        alt.Tooltip("result_type:N", title="type"),
        alt.Tooltip("n:Q", title="count"),
        alt.Tooltip("total:Q", title="total"),
    ],
).properties(width=430, height=480, title="Top 25 terms by total results")

# Direction stripe on the left axis
dir_tick = alt.Chart(top25).mark_tick(thickness=4, size=18).encode(
    y=alt.Y("search_term_lower:N",
            sort=top_order,
            axis=None),
    x=alt.value(0),
    color=alt.Color("dir_label:N",
                    scale=alt.Scale(domain=["LTR", "RTL"],
                                    range=["#1f77b4", "#d62728"]),
                    title="Direction"),
    tooltip=["search_term_lower:N", "dir_label:N"],
).properties(width=18, height=480, title="Direction")

display((dir_tick | top_bar).resolve_scale(y="shared", color="independent"))

# Summary counts
n_ltr = (top25["dir_label"] == "LTR").sum()
n_rtl = (top25["dir_label"] == "RTL").sum()
print(f"Top 25 breakdown: {n_ltr} LTR, {n_rtl} RTL")

alt.HConcatChart(...)

Top 25 breakdown: 23 LTR, 2 RTL


In [11]:
# ── LTR vs RTL comparison ─────────────────────────────────────────────────
dir_summary = (
    hit_dir.groupby("dir_label")
    .agg(
        total_terms=("search_term_lower", "count"),
        terms_with_hits=("any_hit", "sum"),
        total_results=("total", "sum"),
    )
    .reset_index()
)
dir_summary["hit_rate_pct"] = (
    dir_summary["terms_with_hits"] / dir_summary["total_terms"] * 100
).round(1)
dir_summary["avg_per_hit"] = (
    dir_summary["total_results"] / dir_summary["terms_with_hits"].replace(0, pd.NA)
).round(1)

print("LTR vs RTL summary:")
print(dir_summary[["dir_label","total_terms","terms_with_hits","hit_rate_pct",
                    "total_results","avg_per_hit"]].to_string(index=False))

dir_metrics = [
    ("hit_rate_pct",  "Hit rate (%)"),
    ("total_results", "Total results"),
    ("avg_per_hit",   "Avg results / hit term"),
]
dir_charts = []
for col, title in dir_metrics:
    c = alt.Chart(dir_summary).mark_bar().encode(
        x=alt.X("dir_label:N", title=None),
        y=alt.Y(f"{col}:Q", title=title),
        color=alt.Color("dir_label:N",
                        scale=alt.Scale(domain=["LTR", "RTL"],
                                        range=["#1f77b4", "#d62728"]),
                        legend=None),
        tooltip=["dir_label:N", alt.Tooltip(f"{col}:Q", title=title)],
    ).properties(width=130, height=180, title=title)
    dir_charts.append(c)
display(alt.hconcat(*dir_charts))

# ── Top 10 RTL terms ──────────────────────────────────────────────────────
rtl_top = hit_dir[hit_dir["dir_label"] == "RTL"].nlargest(10, "total")
if len(rtl_top):
    print("\nTop 10 RTL terms:")
    print(rtl_top[["search_term_lower","n_repos","n_issues","n_users","total"]].to_string(index=False))
else:
    print("\nNo RTL terms with results.")

# ── Comparison status breakdown ────────────────────────────────────────────────────
# A grouped term may span multiple comparison statuses; take the first for this display.
hit_dir["primary_cat"] = hit_dir["comparison_status"].str.split(",").str[0].str.strip()

cat_summary = (
    hit_dir.groupby("primary_cat")
    .agg(
        total_terms=("search_term_lower", "count"),
        terms_with_hits=("any_hit", "sum"),
        total_results=("total", "sum"),
    )
    .reset_index()
)
cat_summary["hit_rate_pct"] = (
    cat_summary["terms_with_hits"] / cat_summary["total_terms"] * 100
).round(1)
cat_summary = cat_summary.sort_values("total_results", ascending=False)

cat_results = alt.Chart(cat_summary).mark_bar().encode(
    y=alt.Y("primary_cat:N", sort="-x", title=None),
    x=alt.X("total_results:Q", title="total results"),
    color=alt.Color("hit_rate_pct:Q", scale=alt.Scale(scheme="blues"),
                    title="hit rate (%)"),
    tooltip=["primary_cat:N", "total_terms:Q", "terms_with_hits:Q",
             "hit_rate_pct:Q", "total_results:Q"],
).properties(width=300, height=160, title="Total results by comparison_status")

cat_rate = alt.Chart(cat_summary).mark_bar().encode(
    y=alt.Y("primary_cat:N", sort="-x", title=None, axis=alt.Axis(labels=False)),
    x=alt.X("hit_rate_pct:Q", title="hit rate (%)"),
    color=alt.Color("total_terms:Q", scale=alt.Scale(scheme="greys"),
                    title="total terms"),
    tooltip=["primary_cat:N", "total_terms:Q", "hit_rate_pct:Q"],
).properties(width=200, height=160, title="Hit rate by comparison_status")

display(cat_results | cat_rate)
print()
print(cat_summary.to_string(index=False))

LTR vs RTL summary:
dir_label  total_terms  terms_with_hits  hit_rate_pct  total_results  avg_per_hit
      LTR         3882              132           3.4          26318        199.4
      RTL          301                5           1.7            494         98.8


alt.HConcatChart(...)


Top 10 RTL terms:
       search_term_lower  n_repos  n_issues  n_users  total
   humanidades digitales      180        75       46    301
    humanidades digitais       52        50       24    126
                    数位人文        0        65        0     65
      kebudayaan digital        0         1        0      1
     ngữ hóa kỹ thuật số        0         1        0      1
         ڈیجیٹل انسانیات        0         0        0      0
        ڈیجیٹل ہیومینٹیز        0         0        0      0
العلوم الإنسانية الرقمية        0         0        0      0
       الإنسانية الدقيقة        0         0        0      0
      الإنسانيات الرقمية        0         0        0      0


alt.HConcatChart(...)


          primary_cat  total_terms  terms_with_hits  total_results  hit_rate_pct
substantive_variation         4170              127          26018           3.0
    exact_convergence            5                5            688         100.0
near_form_convergence            8                5            106          62.5


## 8.3 — The Value of Multilingual Search

The core methodological question: how much does searching in 800+ languages add over searching in English only, or a manually curated shortlist of major languages?

Three baselines are compared:
- **English only** — search term `"digital humanities"` alone
- **Major 11 languages** — English plus the 10 most-spoken languages in the pipeline (Spanish, French, German, Portuguese, simplified Chinese, Japanese, Korean, Arabic, Russian, Hindi), represented by their ISO 639-1 codes
- **Full multilingual** — all 4,285 kept terms

Because a single GitHub item can be found by multiple search terms, the analysis checks which term(s) found each item across **all** raw rows before deduplication.

In [12]:
MAJOR_LANG_CODES = {"en", "es", "fr", "de", "pt", "zh", "ja", "ko", "ar", "ru", "hi"}

# Build set of search terms that include at least one major language code
major_terms = set()
en_terms    = set()
for _, row in st_kept.iterrows():
    codes = {c.strip() for c in str(row["natural_language"]).split(",") if c.strip()}
    if "en" in codes:
        en_terms.add(row["search_term_lower"])
    if codes & MAJOR_LANG_CODES:
        major_terms.add(row["search_term_lower"])

def value_add(raw_df: pd.DataFrame, id_col: str = "id") -> dict:
    """For each result type, count unique ids reached by each baseline."""
    norm = raw_df["search_term"].str.lower().str.strip()
    en_ids    = set(raw_df[norm.isin(en_terms)][id_col].dropna())
    major_ids = set(raw_df[norm.isin(major_terms)][id_col].dropna())
    all_ids   = set(raw_df[id_col].dropna())
    return {
        "english_only":          len(en_ids),
        "major_11_languages":    len(major_ids),
        "full_multilingual":     len(all_ids),
        "added_over_english":    len(all_ids - en_ids),
        "added_over_major":      len(all_ids - major_ids),
        "english_pct":           round(len(en_ids)    / len(all_ids) * 100, 1) if all_ids else 0,
        "major_pct":             round(len(major_ids) / len(all_ids) * 100, 1) if all_ids else 0,
    }

value_rows = []
for label, raw_df in [("Repositories", repos_raw), ("Issues", issues_raw), ("Users", users_raw)]:
    row = value_add(raw_df)
    row["type"] = label
    value_rows.append(row)

value_df = pd.DataFrame(value_rows).set_index("type")
print(value_df[["english_only","major_11_languages","full_multilingual",
                "added_over_english","added_over_major",
                "english_pct","major_pct"]].to_string())

              english_only  major_11_languages  full_multilingual  added_over_english  added_over_major  english_pct  major_pct
type                                                                                                                           
Repositories          4830                5127               6402                1572              1275         75.4       80.1
Issues                3005                3739               6862                3857              3123         43.8       54.5
Users                 1565                1633               1696                 131                63         92.3       96.3


In [13]:
# Stacked bar: English-only / major-only / multilingual-only split per result type
stacked_rows = []
for label, raw_df in [("Repositories", repos_raw), ("Issues", issues_raw), ("Users", users_raw)]:
    norm      = raw_df["search_term"].str.lower().str.strip()
    en_ids    = set(raw_df[norm.isin(en_terms)]["id"].dropna())
    major_ids = set(raw_df[norm.isin(major_terms)]["id"].dropna())
    all_ids   = set(raw_df["id"].dropna())
    stacked_rows += [
        {"type": label, "segment": "English only",            "n": len(en_ids)},
        {"type": label, "segment": "Major langs, not English", "n": len(major_ids - en_ids)},
        {"type": label, "segment": "Full multilingual only",   "n": len(all_ids - major_ids)},
    ]

seg_order = ["English only", "Major langs, not English", "Full multilingual only"]
stacked_df = pd.DataFrame(stacked_rows)

bar = alt.Chart(stacked_df).mark_bar().encode(
    y=alt.Y("type:N", title=None),
    x=alt.X("n:Q", title="unique items"),
    color=alt.Color("segment:N", sort=seg_order,
                    scale=alt.Scale(range=["#1f77b4", "#ff7f0e", "#2ca02c"]),
                    title="Found by"),
    order=alt.Order("segment:N", sort="ascending"),
    tooltip=["type:N", "segment:N", "n:Q"],
).properties(width=450, height=140,
             title="Unique results by search scope — English only vs. major languages vs. full multilingual")
display(bar)

# Pct labels
for _, row in stacked_df.iterrows():
    total = stacked_df[stacked_df["type"] == row["type"]]["n"].sum()
    print(f"  {row['type']:15s} {row['segment']:35s}: {row['n']:5d} ({row['n']/total*100:.1f}%)")

alt.Chart(...)

  Repositories    English only                       :  4830 (75.4%)
  Repositories    Major langs, not English           :   297 (4.6%)
  Repositories    Full multilingual only             :  1275 (19.9%)
  Issues          English only                       :  3005 (43.8%)
  Issues          Major langs, not English           :   734 (10.7%)
  Issues          Full multilingual only             :  3123 (45.5%)
  Users           English only                       :  1565 (92.3%)
  Users           Major langs, not English           :    68 (4.0%)
  Users           Full multilingual only             :    63 (3.7%)


## 8.4 — Search-Term Overlap and Retrieval Clusters

Counting every term-result link separately overstates the amount of material retrieved, because several translated terms can return the same repository, issue, user, or organization. I therefore compare each term's exclusive results with results shared across terms, then inspect the strongest pairwise overlaps as a network.

Overlap is evidence about retrieval behavior, not proof that two translations are equivalent. A cluster may reflect a genuine multilingual concept, an orthographic variant, GitHub's query normalization, a broad topic match, or an irrelevant phrase. I use the network to identify broader patterns and the HTML relevance reviewer to determine what those patterns mean.

In [14]:
overlap_review = pd.read_csv(SEARCH_RELEVANCE_QUEUE_PATH, keep_default_na=False)
overlap_edges = pd.read_csv(SEARCH_OVERLAP_PATH)
overlap_summary = pd.read_csv(SEARCH_OVERLAP_SUMMARY_PATH)

print("Global retrieval overlap (all productive terms):")
display(overlap_summary.T.rename(columns={0: "count"}))

shared_pct = pd.to_numeric(overlap_review["shared_with_other_terms_pct"], errors="coerce").fillna(0)
overlap_review["overlap_class"] = pd.cut(
    shared_pct,
    bins=[-0.001, 0, 0.10, 0.90, 0.999999, 1.0],
    labels=["Fully exclusive", "Mostly novel", "Mixed", "Mostly redundant", "Fully redundant"],
    include_lowest=True,
)
overlap_order = ["Fully exclusive", "Mostly novel", "Mixed", "Mostly redundant", "Fully redundant"]
overlap_counts = (overlap_review["overlap_class"].value_counts().reindex(overlap_order, fill_value=0)
                  .rename_axis("overlap_class").reset_index(name="terms"))

overlap_bars = alt.Chart(overlap_counts).mark_bar(color="#4472A8").encode(
    y=alt.Y("overlap_class:N", sort=overlap_order, title=None),
    x=alt.X("terms:Q", title="productive translated search terms"),
).properties(width=560, height=220, title={"text": "How much does each search term overlap other terms?",
    "subtitle": "Exclusive/novel terms add distinct results; redundant terms mostly retrieve material found elsewhere."})
overlap_labels = overlap_bars.mark_text(align="left", dx=4, color="black").encode(text="terms:Q")
overlap_chart = overlap_bars + overlap_labels
display(overlap_chart)
save_chart(overlap_chart, str(FIGURE_DIR / "search_term_overlap_distribution.png"), scale_factor=2)

novel_terms = overlap_review.assign(
    new_beyond_english_count=pd.to_numeric(overlap_review["new_beyond_english_count"], errors="coerce").fillna(0).astype(int),
    total_unique_results=pd.to_numeric(overlap_review["total_unique_results"], errors="coerce").fillna(0).astype(int),
    exclusive_result_count=pd.to_numeric(overlap_review["exclusive_result_count"], errors="coerce").fillna(0).astype(int),
).sort_values(["new_beyond_english_count", "exclusive_result_count"], ascending=False)
print("Terms adding the most results beyond the English query (novelty still requires relevance review):")
display(novel_terms[["search_term", "natural_language", "total_unique_results",
                     "exclusive_result_count", "new_beyond_english_count",
                     "shared_with_other_terms_pct"]].head(15))

Global retrieval overlap (all productive terms):


,count
productive_term_count,137
unique_result_count,14960
english_result_count,9187
result_count_beyond_english,5773
exclusive_to_one_term_count,9336
shared_by_two_or_more_terms_count,5624
shared_by_three_or_more_terms_count,3371
shared_by_five_or_more_terms_count,184
maximum_term_multiplicity,14
term_result_link_count,26812


alt.LayerChart(...)

Terms adding the most results beyond the English query (novelty still requires relevance review):


,search_term,natural_language,total_unique_results,exclusive_result_count,new_beyond_english_count,shared_with_other_terms_pct
4,digitäl humäns,vo,1495,1486,1489,0.006020
5,nhân văn số,vi,1018,1014,1018,0.003929
7,数字人文,"dng, gan, hsn, ii, mdr, nxq, smp, wuu, yao, za...",640,576,625,0.100000
8,buku digital,lbw,359,359,359,0.000000
9,humanidades digitales,"arn, ast, ay, bvb, ccr, chb, es, ext, guc, krj...",301,20,266,0.933555
11,电子文学,ii,258,251,258,0.027132
10,humanidades digital,"fil, ilo, kea, pap, tet, tl",283,0,246,1.000000
12,humanités numériques,"bci, en, fr, frc, frm, fro, gcr, lb, ln, mg, n...",248,6,213,0.975806
16,informatica umanistica,it,217,204,207,0.059908
13,humanités numeriques,frp,242,0,207,1.000000


In [15]:
# Retain only substantial, proportionally strong overlaps so large English result sets do not dominate.
network_edges = overlap_edges[(overlap_edges["shared_result_count"] >= 10) &
                              (overlap_edges["jaccard_overlap"] >= 0.40)].copy()
G = nx.from_pandas_edgelist(network_edges, "source_term_key", "target_term_key",
                            edge_attr=["shared_result_count", "jaccard_overlap"])
positions = nx.spring_layout(G, seed=42, k=1.25)
component_by_node = {}
for component_number, component in enumerate(nx.connected_components(G), start=1):
    for node in component:
        component_by_node[node] = f"Cluster {component_number}"

term_totals = overlap_review.set_index("search_term_key")["total_unique_results"].astype(int).to_dict()
network_nodes = pd.DataFrame([{"term": node, "x": positions[node][0], "y": positions[node][1],
                               "cluster": component_by_node[node],
                               "total_unique_results": term_totals.get(node, 0)} for node in G.nodes])
network_links = pd.DataFrame([{"source": row.source_term_key, "target": row.target_term_key,
                               "x": positions[row.source_term_key][0], "y": positions[row.source_term_key][1],
                               "x2": positions[row.target_term_key][0], "y2": positions[row.target_term_key][1],
                               "shared_result_count": row.shared_result_count,
                               "jaccard_overlap": row.jaccard_overlap}
                              for row in network_edges.itertuples(index=False)])

links = alt.Chart(network_links).mark_rule(color="#9A9A9A", opacity=0.55).encode(
    x=alt.X("x:Q", axis=None), y=alt.Y("y:Q", axis=None), x2="x2:Q", y2="y2:Q",
    strokeWidth=alt.StrokeWidth("shared_result_count:Q", scale=alt.Scale(type="sqrt", range=[1, 8]), legend=None),
    tooltip=["source:N", "target:N", "shared_result_count:Q", alt.Tooltip("jaccard_overlap:Q", format=".1%")],
)
points = alt.Chart(network_nodes).mark_circle(stroke="white", strokeWidth=1.2).encode(
    x=alt.X("x:Q", axis=None), y=alt.Y("y:Q", axis=None),
    size=alt.Size("total_unique_results:Q", scale=alt.Scale(type="sqrt", range=[150, 1200]), title="Unique results"),
    color=alt.Color("cluster:N", title="Connected cluster", scale=alt.Scale(scheme="tableau10")),
    tooltip=["term:N", "cluster:N", "total_unique_results:Q"],
)
node_labels = alt.Chart(network_nodes).mark_text(dy=-16, fontSize=10, color="#111111").encode(
    x="x:Q", y="y:Q", text="term:N")
network_chart = (links + points + node_labels).properties(
    width=720, height=500, title={"text": "Strong search-term retrieval clusters",
        "subtitle": "Edges require at least 10 shared results and 40% Jaccard overlap; size shows each term's result set."})
display(network_chart)
save_chart(network_chart, str(FIGURE_DIR / "search_term_overlap_network.png"), scale_factor=2)
print(f"{len(network_nodes)} terms, {len(network_links)} strong overlaps, "
      f"{nx.number_connected_components(G)} connected clusters shown.")

alt.LayerChart(...)

15 terms, 18 strong overlaps, 4 connected clusters shown.


## 8.5 — Could a Simpler Vocabulary Have Found These Results?

The full pipeline is not the only way to assemble multilingual Digital Humanities terminology. Wikipedia labels and conventional translation services offer much cheaper baselines. I therefore treat vocabulary construction as a cumulative ablation: English alone, English plus Wikipedia, all direct sources, and finally the full reviewed pipeline. At each stage I count globally unique GitHub objects rather than summing per-term totals, so an object retrieved by several terms is counted once.

This comparison measures what each source layer made retrievable in this experiment. It does not establish who coined a term. A query classified as **prompt-service only** is an exact string absent from this repository's Wikipedia and direct-service outputs. It may be only an accent or spelling variant of a direct-source term, may be well established in journals or associations, or may be erroneous. I therefore calculate its marginal results beyond the complete direct-source result set and flag accent-insensitive matches separately. A future scholarly-vocabulary baseline is still necessary before I describe any term as genuinely new.

In [16]:
translation_root = Path(DATA_DIR) / "translated_terms" / TERM_SLUG

def normalized_values(path, column):
    if not path.exists():
        return set()
    source_df = pd.read_csv(path, dtype=str, keep_default_na=False, usecols=lambda c: c == column)
    if column not in source_df.columns:
        return set()
    return set(source_df[column].str.strip().str.lower()) - {"", "nan", "none"}

direct_specs = {
    "Wikipedia": ("wikipedia_translations.csv", "wikipedia_translated_term"),
    "Google Translate": ("gt_translations.csv", "gt_translated_term"),
    "EasyNMT": ("enmt_translations.csv", "enmt_translated_term"),
    "Lingvanex": ("lingvanex_translations.csv", "lingvanex_translated_term"),
}
source_term_sets = {
    source: normalized_values(translation_root / "direct_services" / filename, column)
    for source, (filename, column) in direct_specs.items()
}

llm_terms = set()
for path in (translation_root / "prompt_services").glob("*_translations.csv"):
    service = path.name.split("_", 1)[0]
    llm_terms |= normalized_values(path, f"{service}_translated_term")

# Prefix IDs by endpoint so an issue and repository with the same numeric ID remain distinct.
result_links = pd.concat([
    pd.DataFrame({"term_key": repos_raw["search_term"].str.strip().str.lower(),
                  "object_key": "repository:" + repos_raw["id"].astype(str)}),
    pd.DataFrame({"term_key": issues_raw["search_term"].str.strip().str.lower(),
                  "object_key": "issue:" + issues_raw["id"].astype(str)}),
    pd.DataFrame({"term_key": users_raw["search_term"].str.strip().str.lower(),
                  "object_key": "account:" + users_raw["id"].astype(str)}),
], ignore_index=True).drop_duplicates()
productive_terms = set(result_links["term_key"])

english_terms = {TERM.lower()}
wikipedia_terms = source_term_sets["Wikipedia"] & productive_terms
all_direct_terms = set().union(*source_term_sets.values()) & productive_terms
prompt_service_terms = llm_terms & productive_terms
full_terms = productive_terms
stage_terms = [
    ("English only", english_terms & productive_terms),
    ("+ Wikipedia labels", (english_terms | wikipedia_terms) & productive_terms),
    ("+ Other direct MT", (english_terms | all_direct_terms) & productive_terms),
    ("+ Full reviewed pipeline", full_terms),
]

ablation_rows = []
previous_objects = set()
previous_terms = set()
for stage, terms in stage_terms:
    objects = set(result_links.loc[result_links["term_key"].isin(terms), "object_key"])
    ablation_rows.append({
        "stage": stage,
        "productive_terms": len(terms),
        "terms_added": len(terms - previous_terms),
        "unique_results": len(objects),
        "new_results_at_stage": len(objects - previous_objects),
        "pct_of_full_results": len(objects) / len(set(result_links["object_key"])) if len(result_links) else 0,
    })
    previous_objects, previous_terms = objects, terms
ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df.style.format({"pct_of_full_results": "{:.1%}"}))

ablation_long = ablation_df.melt(
    id_vars=["stage"], value_vars=["unique_results", "new_results_at_stage"],
    var_name="measure", value_name="results")
ablation_long["measure"] = ablation_long["measure"].map({
    "unique_results": "Cumulative unique results",
    "new_results_at_stage": "New at this stage",
})
stage_order = [stage for stage, _ in stage_terms]
ablation_chart = alt.Chart(ablation_long).mark_bar().encode(
    y=alt.Y("stage:N", sort=stage_order, title=None),
    x=alt.X("results:Q", title="globally unique GitHub objects"),
    color=alt.Color("measure:N", title=None, scale=alt.Scale(
        domain=["Cumulative unique results", "New at this stage"],
        range=["#4C78A8", "#F28E2B"])),
    column=alt.Column("measure:N", title=None, sort=["Cumulative unique results", "New at this stage"]),
).properties(width=330, height=180, title={"text": "What does each vocabulary source add to GitHub retrieval?",
    "subtitle": "Cumulative ablation; every repository, issue, or account is counted once per stage."})
display(ablation_chart)
save_chart(ablation_chart, str(FIGURE_DIR / "search_vocabulary_source_ablation.png"), scale_factor=2)

direct_baseline_terms = (english_terms | all_direct_terms) & productive_terms
direct_baseline_objects = set(result_links.loc[result_links["term_key"].isin(direct_baseline_terms), "object_key"])

def accent_insensitive_key(value):
    decomposed = unicodedata.normalize("NFKD", str(value))
    return "".join(char for char in decomposed if not unicodedata.combining(char)).lower().strip()

direct_accent_keys = {accent_insensitive_key(term) for term in all_direct_terms | english_terms}
provenance_rows = []
for term in sorted(productive_terms):
    if term in english_terms:
        first_source = "English query"
    elif term in wikipedia_terms:
        first_source = "Wikipedia"
    elif term in all_direct_terms:
        first_source = "Other direct MT"
    elif term in prompt_service_terms:
        first_source = "Prompt-service only"
    else:
        first_source = "Reviewed/corrected pipeline term"
    term_objects = set(result_links.loc[result_links["term_key"].eq(term), "object_key"])
    provenance_rows.append({"search_term": term, "first_available_source": first_source,
                            "unique_results": len(term_objects),
                            "new_beyond_direct_sources": len(term_objects - direct_baseline_objects),
                            "results_already_covered_by_direct": len(term_objects & direct_baseline_objects),
                            "accent_insensitive_direct_match": (term not in all_direct_terms and
                                accent_insensitive_key(term) in direct_accent_keys)})
productive_provenance = pd.DataFrame(provenance_rows)
print("Productive terms by earliest available source in this repository:")
display(productive_provenance.groupby("first_available_source").agg(
    productive_terms=("search_term", "size"),
    summed_term_result_counts=("unique_results", "sum"),
).reset_index())
print("Prompt-service-only exact forms adding the most beyond all direct-source retrieval:")
display(productive_provenance[productive_provenance["first_available_source"].eq("Prompt-service only")]
        .sort_values(["new_beyond_direct_sources", "unique_results"], ascending=False).head(20))
print("Prompt-service-only exact forms that are accent-insensitive matches to direct-source terms:")
display(productive_provenance[(productive_provenance["first_available_source"].eq("Prompt-service only")) &
                              (productive_provenance["accent_insensitive_direct_match"])]
        .sort_values("unique_results", ascending=False))

,stage,productive_terms,terms_added,unique_results,new_results_at_stage,pct_of_full_results
0,English only,1,1,9187,9187,61.4%
1,+ Wikipedia labels,21,20,11136,1949,74.4%
2,+ Other direct MT,34,13,11821,685,79.0%
3,+ Full reviewed pipeline,137,103,14960,3139,100.0%


alt.Chart(...)

Productive terms by earliest available source in this repository:


,first_available_source,productive_terms,summed_term_result_counts
0,English query,1,9187
1,Other direct MT,13,6711
2,Prompt-service only,102,8785
3,Reviewed/corrected pipeline term,1,14
4,Wikipedia,20,2115


Prompt-service-only exact forms adding the most beyond all direct-source retrieval:


,search_term,first_available_source,unique_results,new_beyond_direct_sources,results_already_covered_by_direct,accent_insensitive_direct_match
35,digitäl humäns,Prompt-service only,1495,1488,7,False
3,buku digital,Prompt-service only,359,359,0,False
134,电子文学,Prompt-service only,258,258,0,False
110,ŋɔɔŋ ŋɔɔŋ ŋɔɔŋ ŋɔɔŋ,Prompt-service only,167,167,0,False
1,artes digitales,Prompt-service only,146,140,6,False
120,人工智能文学,Prompt-service only,132,132,0,False
130,數字人文,Prompt-service only,107,89,18,False
132,數字文學,Prompt-service only,71,70,1,False
126,数字文化学,Prompt-service only,52,49,3,False
122,数位人文,Prompt-service only,65,48,17,False


Prompt-service-only exact forms that are accent-insensitive matches to direct-source terms:


,search_term,first_available_source,unique_results,new_beyond_direct_sources,results_already_covered_by_direct,accent_insensitive_direct_match
65,humanités numeriques,Prompt-service only,242,0,242,True
31,digitalná humanistika,Prompt-service only,6,0,6,True


## 8.6 — Language and Family Coverage

How many pipeline languages have at least one result? And which languages contribute uniquely — i.e., results that would not have been found without that language's translation?

In [17]:
# For each result type: map raw rows → individual language codes via lang_term_map
def attach_languages(raw_df: pd.DataFrame) -> pd.DataFrame:
    """Join raw results to individual language codes via the search-term map."""
    norm = raw_df.copy()
    norm["search_term_lower"] = norm["search_term"].str.lower().str.strip()
    return norm.merge(
        lang_term_map[["search_term_lower", "language_code", "language_name", "language_family"]],
        on="search_term_lower", how="left",
    )

repos_lang  = attach_languages(repos_raw)
issues_lang = attach_languages(issues_raw)
users_lang  = attach_languages(users_raw)

# Languages with ≥1 result per type
for label, df in [("Repos", repos_lang), ("Issues", issues_lang), ("Users", users_lang)]:
    n = df["language_code"].dropna().nunique()
    print(f"{label}: {n}/880 languages with ≥1 result ({n/880*100:.1f}%)")

# Any type
all_lang_codes = (
    pd.concat([repos_lang, issues_lang, users_lang])["language_code"]
    .dropna().unique()
)
print(f"Any type: {len(all_lang_codes)}/880 languages with ≥1 result ({len(all_lang_codes)/880*100:.1f}%)")

Repos: 196/880 languages with ≥1 result (22.3%)
Issues: 235/880 languages with ≥1 result (26.7%)
Users: 78/880 languages with ≥1 result (8.9%)
Any type: 277/880 languages with ≥1 result (31.5%)


In [18]:
# Per-language repo count — top 30
lang_repo_counts = (
    repos_lang.groupby(["language_code", "language_name", "language_family"])["id"]
    .nunique()
    .reset_index(name="n_repos")
    .sort_values("n_repos", ascending=False)
)
print("Top 30 languages by unique repo count:")
print(lang_repo_counts.head(30).to_string(index=False))

Top 30 languages by unique repo count:
language_code            language_name                 language_family  n_repos
           en                  English         Indo-European languages     4830
           id               Indonesian          Austronesian languages     2085
          nia                     Nias          Austronesian languages     2079
          gor                Gorontalo          Austronesian languages     2079
          mwv                 Mentawai          Austronesian languages     2078
          dtp            Central Dusun          Austronesian languages     2067
          rob                     Tae'          Austronesian languages     2066
          iba                     Iban          Austronesian languages     2064
           ms                    Malay          Austronesian languages     2063
           vo                  Volapük            Artificial languages      666
          frm            Middle French         Indo-European languages      271
 

In [19]:
# Languages that contribute results NOT found by English
en_repo_ids = set(repos_raw[repos_raw["search_term"].str.lower().str.strip().isin(en_terms)]["id"].dropna())

unique_by_lang = (
    repos_lang[~repos_lang["id"].isin(en_repo_ids)]
    .groupby(["language_code", "language_name", "language_family"])["id"]
    .nunique()
    .reset_index(name="n_unique_repos")
    .sort_values("n_unique_repos", ascending=False)
)
print(f"Languages contributing repos NOT found by English: {len(unique_by_lang)}")
print()
print("Top 20:")
print(unique_by_lang.head(20).to_string(index=False))

Languages contributing repos NOT found by English: 186

Top 20:
language_code            language_name         language_family  n_unique_repos
           vo                  Volapük    Artificial languages             662
          lbw                   Tolaki  Austronesian languages             236
          lad   Ladino / Judeo-Spanish Indo-European languages             211
          mwl                Mirandese Indo-European languages             211
          ext             Extremaduran Indo-European languages             211
          kea             Kabuverdianu     Creoles and pidgins             208
          tet                    Tetum  Austronesian languages             208
          ast                 Asturian Indo-European languages             170
          guc                    Wayuu                Arawakan             169
          yua                 Yucateco                   Mayan             169
          rap                  Rapanui  Austronesian languages     

## 8.7 — Result Distribution by Language Family

Aggregating results to the language family level shows which parts of the world's DH activity are captured by the multilingual approach, and which families only appear because of the full translation coverage (i.e., they would be missed by English-only or major-language search).

In [20]:
# Unique repos per language family
fam_repos = (
    repos_lang.groupby("language_family")["id"]
    .nunique()
    .reset_index(name="n_repos")
    .dropna(subset=["language_family"])
    .sort_values("n_repos", ascending=False)
)

# Flag which families are reachable via English / major languages
en_families    = set(repos_lang[repos_lang["id"].isin(en_repo_ids)]["language_family"].dropna())
major_repo_ids = set(repos_raw[repos_raw["search_term"].str.lower().str.strip().isin(major_terms)]["id"].dropna())
major_families = set(repos_lang[repos_lang["id"].isin(major_repo_ids)]["language_family"].dropna())

fam_repos["reachable_by_english"] = fam_repos["language_family"].isin(en_families)
fam_repos["reachable_by_major"]   = fam_repos["language_family"].isin(major_families)
fam_repos["multilingual_only"]    = ~fam_repos["reachable_by_major"]

# Compute color comparison_status in pandas — nested alt.condition not supported in Altair v6
def reach_label(row):
    if row["multilingual_only"]:
        return "Multilingual only"
    if row["reachable_by_english"]:
        return "English"
    return "Major langs"

fam_repos["reach"] = fam_repos.apply(reach_label, axis=1)

fam_bar = alt.Chart(fam_repos).mark_bar().encode(
    y=alt.Y("language_family:N", sort="-x", title=None),
    x=alt.X("n_repos:Q", title="unique repositories"),
    color=alt.Color("reach:N",
                    scale=alt.Scale(
                        domain=["English", "Major langs", "Multilingual only"],
                        range=["#1f77b4", "#ff7f0e", "#2ca02c"],
                    ),
                    title="Reachable by"),
    tooltip=["language_family:N", "n_repos:Q", "reach:N"],
).properties(width=400, height=380,
             title="Unique repos per language family — blue=English, orange=major langs, green=multilingual only")
display(fam_bar)

print(f"Families reachable by English only        : {len(en_families)}")
print(f"Families reachable by major 11 languages  : {len(major_families)}")
print(f"Families only reachable via full pipeline : {fam_repos['multilingual_only'].sum()}")
if fam_repos["multilingual_only"].any():
    print("\nMultilingual-only families:")
    for _, r in fam_repos[fam_repos["multilingual_only"]].iterrows():
        print(f"  {r['language_family']}: {r['n_repos']} repos")

alt.Chart(...)

Families reachable by English only        : 21
Families reachable by major 11 languages  : 27
Families only reachable via full pipeline : 5

Multilingual-only families:
  Nuclear-Macro-Je: 6 repos
  Mande: 2 repos
  Kru: 2 repos
  Khoe-Kwadi: 1 repos
  Albanian languages: 1 repos


In [21]:
# Cross-type summary: repos + issues + users per family
fam_issues = (
    issues_lang.groupby("language_family")["id"]
    .nunique().reset_index(name="n_issues").dropna(subset=["language_family"])
)
fam_users = (
    users_lang.groupby("language_family")["id"]
    .nunique().reset_index(name="n_users").dropna(subset=["language_family"])
)

fam_all = (
    fam_repos[["language_family","n_repos"]]
    .merge(fam_issues, on="language_family", how="outer")
    .merge(fam_users,  on="language_family", how="outer")
    .fillna(0)
)
for col in ["n_repos","n_issues","n_users"]:
    fam_all[col] = fam_all[col].astype(int)
fam_all["total"] = fam_all["n_repos"] + fam_all["n_issues"] + fam_all["n_users"]
fam_all = fam_all.sort_values("total", ascending=False)

print("Results by language family (all types):")
print(fam_all.to_string(index=False))

Results by language family (all types):
                  language_family  n_repos  n_issues  n_users  total
          Indo-European languages     5342      3296     1687  10325
           Austronesian languages     2773      3827       97   6697
                   Atlantic-Congo      462      1177       75   1714
             Artificial languages      667       878        0   1545
           Sino-Tibetan languages       91      1088        0   1179
           Afro-Asiatic languages      304       737       70   1111
         Austro-Asiatic languages        0      1042        0   1042
              Tai-Kadai languages       75       652        0    727
              Creoles and pidgins      393       228       73    694
                         Arawakan      234       140       70    444
                      Uto-Aztecan      180        75       46    301
                      Otomanguean      180        75       46    301
                       Misumalpan      180        75       46  